# 02 — Train the semantic consistency classifier

Fine-tunes Bio_ClinicalBERT with LoRA (Section 3.2.3) to judge whether a claim's clinical
notes are consistent with its billed ICD-11 code, on a Colab GPU. Results and checkpoints
go to Google Drive.

| Step | Module | Output (in Drive) |
|---|---|---|
| Export | `src.classification.export` | `/content/classifier_data/*.jsonl` (temporary) |
| Train | `src.classification.train` | `experiments/classifier/<version>/<run_id>/`: LoRA adapter, `config.json`, `metrics.json` |

**Before running:** Runtime → Change runtime type → T4 GPU. Drive must hold
`MyDrive/claimguard/.env` (`GITHUB_TOKEN=...`, optionally `HF_TOKEN=...`),
`data/processed/<version>/` and `artifacts/icd11_cache.json`.

Only the train and validation splits are used. The test split stays untouched until the
final evaluation (Section 3.2.4).

In [11]:
DATASET_VERSION = 'v0.5-5k'
BRANCH = 'dev'
BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'   # fallback: 'bert-base-uncased'
SEEDS = [42]        # one run for the core workflow; all five in Sprint 6: [42, 123, 2024, 7, 31415]
EPOCHS = 5

## 1. Bootstrap: Drive, repository, paths

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
CACHE = f'{ROOT}/artifacts/icd11_cache.json'
EXPERIMENTS = f'{ROOT}/experiments/classifier/{DATASET_VERSION}'
ENV_LOGS = f'{ROOT}/artifacts/environment_logs'
EXPORT = '/content/classifier_data'   # derived from PROCESSED + CACHE, so not kept in Drive
for path in (f'{PROCESSED}/train.parquet', CACHE):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} missing: upload it to Drive first')
for d in (EXPERIMENTS, ENV_LOGS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
commit 67a0c2fbae547ee183507229e4b2f46b4d50b191


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip install -q peft
# Colab preinstalls torchao 0.10, which this project doesn't use; peft >= 0.21 refuses to
# start LoRA when an old torchao is present ("only versions above 0.16.0 are supported").
!pip uninstall -q -y torchao
import torch, transformers, peft
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__,
      '| cuda', torch.cuda.is_available())

## 2. Export the splits with ICD-11 reference text

In [14]:
!python -m src.classification.export --data {PROCESSED} --cache {CACHE} --out {EXPORT}

train: 2700 examples
validation: 900 examples
test: 900 examples
deployment_pool: 500 examples


## 3. Train (a few minutes per seed on a T4)

The checkpoint with the best validation F1 is kept; its metrics are what gets reported.

In [15]:
seeds = ' '.join(map(str, SEEDS))
!python -m src.classification.train --train-file {EXPORT}/train.jsonl --val-file {EXPORT}/validation.jsonl \
    --base-model {BASE_MODEL} --experiments-dir {EXPERIMENTS} --seeds {seeds} --epochs {EPOCHS}

device: cuda; 2700 train, 900 validation examples
config.json: 100% 385/385 [00:00<00:00, 2.83MB/s]
vocab.txt: 213kB [00:00, 20.1MB/s]
pytorch_model.bin: 100% 436M/436M [00:02<00:00, 169MB/s] 
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/claimguard/src/classification/train.py", line 214, in <module>
    results = train_all_seeds(train_ex, val_ex, args.base_model, args.seeds, Path(args.experiments_dir),
                              epochs=args.epochs, batch_size=args.batch_size, lr=args.lr,
                              patience=args.patience, max_length=args.max_length,
   

## 4. Results and run record

In [16]:
import json, glob, datetime
runs = sorted(glob.glob(f'{EXPERIMENTS}/*/config.json'))
for path in runs[-len(SEEDS):]:
    record = json.load(open(path))
    record['colab_commit'] = COMMIT
    json.dump(record, open(path, 'w'), indent=2)
    m = record['metrics']
    print(f"{record['run_id']}: best epoch {m['epoch']}, precision {m['precision']:.3f}, "
          f"recall {m['recall']:.3f}, f1 {m['f1']:.3f} ({m['flagged']} flagged, {m['positives']} inconsistent)")
stamp = datetime.date.today().isoformat()
!pip freeze > {ENV_LOGS}/requirements_frozen_{stamp}_classifier.txt